In [ ]:
import os
import re
import json
import zipfile
import hashlib
import random
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd


# =========================
# 0) 사용자 설정(여기만 수정)
# =========================
ZIP_DIR = "../data/raw/clickbait_labels"          # zip 42개가 들어있는 폴더
OUT_CSV = "../data/impact/impact3_dataset_500.csv"
OUT_JSON = "../data/impact/impact3_dataset_500.json"
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
TARGET_N = 500

# 데이터에서 정치 표기가 "정치"가 아닐 가능성까지 고려(필요 시 추가)
POLITICS_CATS = {"정치"}  # 예: {"정치", "PO"} 등으로 확장 가능

# 샘플링 재현성
RANDOM_SEED = 42


# =========================
# 1) 유틸: 텍스트 정리/문장 추출
# =========================
CAPTION_PAT = re.compile(r"^(사진|자료|그래픽|캡처|출처)\s*[:=]|^ⓒ|^<|^\(|^\[", re.IGNORECASE)
REPORTER_PAT = re.compile(r"(기자\s*=|기자\s*[:=]|연합뉴스|뉴스1|특파원|메일|@)", re.IGNORECASE)

def normalize_ws(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_sentences_ko(text: str) -> List[str]:
    """
    단순 규칙 기반 문장 분리:
    - 마침표/물음표/느낌표/…/다./요. 등 종결을 기준으로 분리
    """
    t = normalize_ws(text)
    if not t:
        return []

    # 종결 후보에서 분리되도록 구분자 추가
    # '다.' '요.' 같은 종결도 분리 대상으로 처리
    t = re.sub(r"([.!?…])\s+", r"\1\n", t)
    t = re.sub(r"(다\.|요\.)\s+", r"\1\n", t)

    parts = [p.strip() for p in t.split("\n") if p.strip()]
    return parts

def pick_meaningful_sentence(content: str, min_len: int = 15) -> str:
    """
    본문에서 '의미 있는' 1문장만 선택.
    - 캡션성/기자정보/너무 짧은 문장은 스킵
    - 첫 유효 문장을 반환
    """
    sentences = split_sentences_ko(content)

    for s in sentences:
        s2 = normalize_ws(s)
        if len(s2) < min_len:
            continue
        if CAPTION_PAT.search(s2):
            continue
        # 기자/메일/통신사 표시만 있는 문장 배제(완전 배제는 과하므로 "강한 힌트"만)
        if REPORTER_PAT.search(s2) and len(s2) < 40:
            continue
        return s2

    # 모두 탈락이면, 그래도 첫 문장이 있으면 최대한 반환(완전 공백은 방지)
    return normalize_ws(sentences[0]) if sentences else ""


# =========================
# 2) 유틸: JSON 레코드 파싱
# =========================
def iter_json_records_from_bytes(b: bytes) -> Iterable[Dict[str, Any]]:
    """
    zip 내부 파일이
    - JSON 배열([{}, {}])이거나
    - 단일 JSON 객체({})이거나
    - JSON Lines({}\n{}\n...) 형태일 수 있으므로 모두 대응
    """
    txt = b.decode("utf-8", errors="ignore").strip()
    if not txt:
        return

    # JSON Lines 추정: 첫 글자가 '{'이고 중간에 줄바꿈으로 다수 객체가 이어지는 형태
    if txt.startswith("{") and "\n{" in txt:
        for line in txt.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    yield obj
            except Exception:
                continue
        return

    # 일반 JSON
    try:
        obj = json.loads(txt)
    except Exception:
        return

    if isinstance(obj, dict):
        yield obj
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict):
                yield item


def extract_fields(rec: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    AI Hub 낚시성 기사 탐지 데이터 구조에서 필요한 필드만 뽑기.
    일반적으로 sourceDataInfo 아래에 있음.
    """
    sdi = rec.get("sourceDataInfo") or rec.get("source_data_info") or rec

    # 필드 후보들(데이터셋 변형 대비)
    news_id = sdi.get("newsID") or sdi.get("newsId") or sdi.get("id")
    cat = sdi.get("newsCategory") or sdi.get("category")
    sub = sdi.get("newsSubcategory") or sdi.get("subcategory") or ""
    title = sdi.get("newsTitle") or sdi.get("title")
    content = sdi.get("newsContent") or sdi.get("content")
    use_type = sdi.get("useType")

    if cat is None or title is None or content is None or use_type is None:
        return None

    # 문자열 정리
    cat = str(cat).strip()
    sub = str(sub).strip() if sub is not None else ""
    title = normalize_ws(str(title))
    content = normalize_ws(str(content))

    # useType이 문자열로 들어오는 경우 대비
    try:
        use_type = int(use_type)
    except Exception:
        return None

    return {
        "newsID": str(news_id).strip() if news_id is not None else None,
        "newsCategory": cat,
        "newsSubcategory": sub,
        "newsTitle": title,
        "newsContent_raw": content,
        "useType": use_type,
    }


# =========================
# 3) 메인: zip 폴더 순회 -> 필터 -> 정제 -> 500개 샘플링 -> 저장
# =========================
def md5(s: str) -> str:
    return hashlib.md5(s.encode("utf-8", errors="ignore")).hexdigest()

def load_candidates(zip_dir: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    for fn in os.listdir(zip_dir):
        if not fn.lower().endswith(".zip"):
            continue
        zpath = os.path.join(zip_dir, fn)

        with zipfile.ZipFile(zpath, "r") as z:
            for member in z.namelist():
                # 보통 .json / .JSON이 많음
                if not member.lower().endswith(".json"):
                    continue
                try:
                    b = z.read(member)
                except Exception:
                    continue

                for rec in iter_json_records_from_bytes(b):
                    f = extract_fields(rec)
                    if not f:
                        continue

                    # IMPACT 3 필터: 정치 + 낚시성(useType=0)
                    if f["newsCategory"] not in POLITICS_CATS:
                        continue
                    if f["useType"] != 0:
                        continue
                    if not f["newsTitle"] or not f["newsContent_raw"]:
                        continue

                    # 본문 1문장 추출
                    one_sent = pick_meaningful_sentence(f["newsContent_raw"])
                    if not one_sent:
                        continue

                    rows.append({
                        "newsID": f["newsID"],
                        "newsCategory": f["newsCategory"],
                        "newsSubcategory": f["newsSubcategory"],
                        "newsTitle": f["newsTitle"],
                        "newsContent": one_sent,     # 최종 저장용: 1문장
                        "useType": f["useType"],     # 원천 값 그대로 (여기선 0만 남음)
                    })

    return rows

def dedup_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen_id = set()
    seen_text = set()
    out = []

    for r in rows:
        nid = r.get("newsID")
        key_text = md5((r["newsTitle"] + "||" + r["newsContent"]).strip())

        # newsID가 있으면 우선 중복 제거
        if nid:
            if nid in seen_id:
                continue
            seen_id.add(nid)

        # title+1문장 본문 완전중복 제거
        if key_text in seen_text:
            continue
        seen_text.add(key_text)

        out.append(r)

    return out

def stratified_sample_by_subcat(rows: List[Dict[str, Any]], n: int, seed: int = 42) -> List[Dict[str, Any]]:
    """
    newsSubcategory 쏠림을 완화하기 위한 간단한 층화 샘플링.
    - 가능한 한 subcategory별로 균등하게 뽑되,
      부족한 subcategory는 있는 만큼만 뽑고 남는 quota는 다른 그룹에서 채움.
    """
    random.seed(seed)

    # 그룹핑
    groups: Dict[str, List[Dict[str, Any]]] = {}
    for r in rows:
        k = r.get("newsSubcategory") or ""
        groups.setdefault(k, []).append(r)

    # 섞기
    for k in groups:
        random.shuffle(groups[k])

    subcats = list(groups.keys())
    if not subcats:
        return []

    base = n // len(subcats)
    rem = n % len(subcats)

    picked: List[Dict[str, Any]] = []

    # 1차: 기본 할당
    for k in subcats:
        take = min(base, len(groups[k]))
        picked.extend(groups[k][:take])
        groups[k] = groups[k][take:]

    # 2차: 나머지 할당(남아있는 그룹에서)
    # rem을 먼저 채우되, 부족하면 전체에서 추가
    # rem 채우기
    while rem > 0:
        progressed = False
        for k in subcats:
            if rem <= 0:
                break
            if groups[k]:
                picked.append(groups[k].pop(0))
                rem -= 1
                progressed = True
        if not progressed:
            break

    # 3차: 아직 부족하면 남은 전체에서 채움
    if len(picked) < n:
        pool = []
        for k in subcats:
            pool.extend(groups[k])
        random.shuffle(pool)
        need = n - len(picked)
        picked.extend(pool[:need])

    return picked[:n]

def main():
    rows = load_candidates(ZIP_DIR)
    print(f"[1] 후보 수(필터 후): {len(rows)}")

    rows = dedup_rows(rows)
    print(f"[2] 중복 제거 후: {len(rows)}")

    if len(rows) < TARGET_N:
        raise RuntimeError(f"후보가 {TARGET_N}개보다 적습니다. 현재: {len(rows)}개. "
                           f"POLITICS_CATS 또는 원천 데이터 구조/표기값을 확인하세요.")

    # 하위카테고리 쏠림 완화 샘플링(원치 않으면 random.sample로 교체 가능)
    sampled = stratified_sample_by_subcat(rows, TARGET_N, seed=RANDOM_SEED)
    print(f"[3] 샘플링 결과: {len(sampled)}")

    # 최종 저장 스키마(요청 5개 컬럼만)
    df = pd.DataFrame(sampled)[["newsCategory", "newsSubcategory", "newsTitle", "newsContent", "useType"]]

    # CSV 저장
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    # JSON 저장(리스트 형태)
    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(df.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

    print(f"[DONE] CSV 저장: {OUT_CSV}")
    print(f"[DONE] JSON 저장: {OUT_JSON}")
    print(df.head(3))

if __name__ == "__main__":
    main()
